# Support Vector Regression (SVR)

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 2.5 hours | **Week 10 — Notebook 9**

---

## Why Does This Matter?

Standard regression (OLS/linear) minimizes the sum of squared errors — every prediction error, no matter how small, contributes to the loss. This makes the model sensitive to noise: tiny fluctuations in the data still "pull" the regression line. SVR fixes this by defining a **tolerance tube** around the prediction. Errors inside the tube are free; only errors outside the tube are penalized. The result is a regression model that:

- Is **robust to small noise** (free zone absorbs it)
- Is **robust to outliers** (large C needed to overfit them)
- Uses a **sparse solution** — only tube-violating points determine the model
- Naturally extends to **nonlinear regression** via the kernel trick

**Real-world use:** Predicting email response time, financial forecasting, load prediction in power grids — any domain where you want a model that doesn't obsess over tiny errors.

## Real-World Analogy: The Plumber's Tolerance Tube

Imagine a plumber who quotes prices for home repair jobs. They have a **±\$100 tolerance**: any project that comes in within \$100 of the estimate is considered "on budget" — no penalty, no stress. 

But if a project runs **more than \$100 over or under**, the plumber gets penalized proportional to how far outside the budget they are.

| Scenario | Actual vs. Estimate | Penalty |
|----------|--------------------|---------|
| Project A | \$50 over | 0 (inside ±\$100 tube) |
| Project B | \$80 under | 0 (inside ±\$100 tube) |
| Project C | \$150 over | \$50 (150 - 100) |
| Project D | \$300 over | \$200 (300 - 100) |

**SVR works the same way:**
- The "tolerance" is **ε (epsilon)** — the tube half-width
- Predictions within ε of the true value → **zero loss**
- Predictions more than ε away → **linear penalty** proportional to the overshoot
- The plumber's skill (the regression function) is judged only by projects outside the tolerance — inside-tube projects don't shape how they quote next time

## Plain English Concept

**Standard regression** says: "Be as close as possible to every single training point. Even a 0.001 error matters."

**SVR says:** "Give me a tube of width 2ε centered on the regression function. I only care about points outside the tube. Find the flattest (simplest) function that keeps as many points inside the tube as possible."

The key insight: **flattest** means the weight vector **w** has the smallest possible norm (||w||²). A flat function generalizes better — it doesn't overfit to the training data's noise.

The tradeoff parameter **C** controls how strict you are about tube violations:
- **Large C**: "Tube violations are unacceptable — fit the training data tightly"
- **Small C**: "Some tube violations are OK — keep the function smooth"

---

## The ε-Insensitive Loss Function

$$L_\varepsilon(y, f(x)) = \max(0, |y - f(x)| - \varepsilon)$$

- If $|y - f(x)| \leq \varepsilon$: **loss = 0** (inside tube)
- If $|y - f(x)| > \varepsilon$: **loss = |error| - ε** (outside tube)

Compare to MSE loss: $L_{MSE}(y, f(x)) = (y - f(x))^2$ — always positive, quadratic growth

---

## SVR Primal Formulation

$$f(x) = \mathbf{w} \cdot \mathbf{x} + b$$

**Minimize:**
$$\frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} (\xi_i + \xi_i^*)$$

**Subject to:**
$$y_i - (\mathbf{w} \cdot \mathbf{x}_i + b) \leq \varepsilon + \xi_i \quad \text{(upper tube violation)}$$
$$( \mathbf{w} \cdot \mathbf{x}_i + b) - y_i \leq \varepsilon + \xi_i^* \quad \text{(lower tube violation)}$$
$$\xi_i, \xi_i^* \geq 0$$

Where:
- $\xi_i$ = slack for points **above** the tube (actual > prediction + ε)
- $\xi_i^*$ = slack for points **below** the tube (actual < prediction - ε)
- **Support vectors** = points with $\xi_i > 0$ or $\xi_i^* > 0$ (outside the tube)

---

## SVR Support Vectors in Regression

Unlike classification SVMs (where SVs are on the margin), regression SVs are:
- Points **outside** the ε-tube: they violate the tube → $\alpha_i \neq 0$ → they contribute to w
- Points **inside** the tube: zero weight → $\alpha_i = 0$ → they have no influence on the model

This sparsity is a powerful property: in a noisy dataset with 1000 points, maybe only 50 are outside the tube, and only those 50 points determine the regression function.

In [ ]:
# =============================================================================
# SECTION 1: Imports & Setup
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')

print("Libraries loaded. Ready for SVR exploration.")
print("Theme: Email response time prediction")

In [ ]:
# =============================================================================
# SECTION 2: Email Response Time Dataset
# =============================================================================
# Scenario: We work at a tech company. We want to predict how long it will
# take an employee to respond to an email (in hours). Features:
#   - word_count: number of words in the email
#   - complexity_score: readability / technical complexity (0-10)
#   - sender_importance: importance level of sender (0-10)

np.random.seed(42)
n_samples = 300

word_count       = np.random.uniform(50, 500, n_samples)       # 50-500 words
complexity_score = np.random.uniform(1, 10, n_samples)         # 1-10
sender_importance = np.random.uniform(1, 10, n_samples)        # 1-10

# True response time: complex, important emails take longer
# Long emails also take longer to respond to
noise = np.random.normal(0, 1.5, n_samples)
response_time = (
    0.01 * word_count +
    0.5 * complexity_score -
    0.3 * sender_importance +   # important senders get faster responses
    3.0 +                        # base response time (hours)
    noise
)
response_time = np.clip(response_time, 0.5, 20)  # clamp to realistic range

# Build DataFrame
df = pd.DataFrame({
    'word_count':        word_count,
    'complexity_score':  complexity_score,
    'sender_importance': sender_importance,
    'response_time':     response_time
})

print("Email Response Time Dataset")
print("="*45)
print(df.describe().round(2))
print(f"\nSamples: {len(df)}")
print(f"Response time range: {df.response_time.min():.2f} – {df.response_time.max():.2f} hours")

In [ ]:
# =============================================================================
# SECTION 3: Visualize the ε-Insensitive Loss vs MSE Loss
# =============================================================================
# Before training anything, let's understand the LOSS FUNCTION that SVR optimizes.

errors = np.linspace(-4, 4, 300)   # hypothetical prediction errors
epsilon = 1.0

# MSE loss: always positive, quadratic
mse_loss = errors**2

# MAE loss: absolute value (for comparison)
mae_loss = np.abs(errors)

# Epsilon-insensitive loss: zero inside tube, linear outside
eps_loss = np.maximum(0, np.abs(errors) - epsilon)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: All three loss functions ---
ax = axes[0]
ax.plot(errors, mse_loss,  label='MSE Loss',                    color='#e74c3c', linewidth=2.5)
ax.plot(errors, mae_loss,  label='MAE Loss',                    color='#f39c12', linewidth=2.5, linestyle='--')
ax.plot(errors, eps_loss,  label=f'ε-Insensitive (ε={epsilon})', color='#2ecc71', linewidth=2.5)
ax.axvspan(-epsilon, epsilon, alpha=0.15, color='#2ecc71', label='Free zone (loss=0)')
ax.axvline(-epsilon, color='#2ecc71', linestyle=':', linewidth=1.5)
ax.axvline(+epsilon, color='#2ecc71', linestyle=':', linewidth=1.5)
ax.set_xlabel('Prediction Error  (y − f(x))', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Functions: MSE vs MAE vs ε-Insensitive', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(-0.3, 6)

# --- Right: Zoom in on ε-insensitive, annotated ---
ax2 = axes[1]
ax2.plot(errors, eps_loss, color='#2ecc71', linewidth=3, label=f'ε-Insensitive Loss (ε={epsilon})')
ax2.axvspan(-epsilon, epsilon, alpha=0.2, color='#2ecc71')
ax2.axvline(-epsilon, color='#27ae60', linestyle='--', linewidth=2)
ax2.axvline(+epsilon, color='#27ae60', linestyle='--', linewidth=2)

# Annotate the flat zone
ax2.annotate('Free zone\n(loss = 0)', xy=(0, 0.05), fontsize=11,
             ha='center', color='#27ae60', fontweight='bold')
ax2.annotate('Penalty = |error| − ε', xy=(2.5, 1.2), fontsize=10,
             ha='center', color='#e74c3c',
             arrowprops=dict(arrowstyle='->', color='#e74c3c'),
             xytext=(2.5, 1.8))
ax2.annotate('Penalty = |error| − ε', xy=(-2.5, 1.2), fontsize=10,
             ha='center', color='#e74c3c',
             arrowprops=dict(arrowstyle='->', color='#e74c3c'),
             xytext=(-2.5, 1.8))

ax2.set_xlabel('Prediction Error  (y − f(x))', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('ε-Insensitive Loss: Annotated', fontsize=13, fontweight='bold')
ax2.set_ylim(-0.2, 3.0)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.suptitle('Understanding the SVR Loss Function', fontsize=15, fontweight='bold', y=1.02)
plt.show()

print("Key insight:")
print("  MSE loss: always penalizes even tiny errors (quadratically!)")
print(f"  ε-insensitive: errors within ±{epsilon} cost NOTHING. Outside → linear penalty.")
print("  This makes SVR robust to small noise and sparse (only border/outside points matter).")

In [ ]:
# =============================================================================
# SECTION 4: SVR with Linear Kernel — Effect of ε (1D view)
# =============================================================================
# For visualization, we use a 1D version of the problem:
# predict response_time from word_count only.

np.random.seed(42)

# 1D dataset: word_count -> response_time (subset for clarity)
X_1d = word_count.reshape(-1, 1)
y_1d = response_time

scaler = StandardScaler()
X_1d_scaled = scaler.fit_transform(X_1d)

# Sort for plotting
sort_idx = np.argsort(X_1d_scaled.ravel())
X_plot = X_1d_scaled[sort_idx]
y_plot = y_1d[sort_idx]
X_grid = np.linspace(X_1d_scaled.min(), X_1d_scaled.max(), 300).reshape(-1, 1)

epsilons = [0.1, 0.5, 2.0]
colors   = ['#3498db', '#e67e22', '#9b59b6']

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, eps, col in zip(axes, epsilons, colors):
    svr = SVR(kernel='linear', epsilon=eps, C=1.0)
    svr.fit(X_1d_scaled, y_1d)
    y_pred_grid = svr.predict(X_grid)

    # Identify support vectors (tube-violating points)
    y_pred_all = svr.predict(X_1d_scaled)
    residuals  = np.abs(y_1d - y_pred_all)
    inside_tube = residuals <= eps + 1e-6
    n_sv = (~inside_tube).sum()

    # Plot
    ax.scatter(X_1d_scaled[inside_tube],  y_1d[inside_tube],
               color='#95a5a6', alpha=0.5, s=20, label='Inside tube')
    ax.scatter(X_1d_scaled[~inside_tube], y_1d[~inside_tube],
               color=col, alpha=0.85, s=50, edgecolors='black', linewidths=0.7,
               label=f'Support Vectors ({n_sv})', zorder=5)

    ax.plot(X_grid, y_pred_grid, color=col, linewidth=2.5, label='SVR line')
    ax.fill_between(X_grid.ravel(),
                    y_pred_grid - eps, y_pred_grid + eps,
                    alpha=0.2, color=col, label=f'ε-tube (±{eps})')

    ax.set_xlabel('Word Count (scaled)', fontsize=11)
    ax.set_title(f'ε = {eps}  |  SVs = {n_sv}', fontsize=13, fontweight='bold', color=col)
    ax.legend(fontsize=9, loc='upper left')

axes[0].set_ylabel('Response Time (hours)', fontsize=11)
plt.suptitle('SVR with Linear Kernel — Effect of ε on Tube Width and Support Vectors',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observation:")
for eps, col in zip(epsilons, colors):
    svr = SVR(kernel='linear', epsilon=eps, C=1.0)
    svr.fit(X_1d_scaled, y_1d)
    y_pred_all = svr.predict(X_1d_scaled)
    residuals  = np.abs(y_1d - y_pred_all)
    n_sv = (residuals > eps + 1e-6).sum()
    pct_inside = 100 * (residuals <= eps + 1e-6).mean()
    print(f"  ε={eps:4.1f} → {n_sv:3d} support vectors, {pct_inside:.1f}% of data inside tube")

In [ ]:
# =============================================================================
# SECTION 5: Effect of C — Fit Tightness
# =============================================================================
# C controls the PENALTY for tube violations.
# Large C → penalize violations heavily → tight fit to data
# Small C → allow more violations → smoother / flatter function

C_values = [0.1, 1.0, 10.0, 100.0]
eps_fixed = 0.5

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
axes = axes.ravel()

palette = ['#1abc9c', '#3498db', '#e74c3c', '#8e44ad']

for ax, C, col in zip(axes, C_values, palette):
    svr = SVR(kernel='linear', epsilon=eps_fixed, C=C)
    svr.fit(X_1d_scaled, y_1d)
    y_pred_grid = svr.predict(X_grid)
    y_pred_all  = svr.predict(X_1d_scaled)

    residuals   = np.abs(y_1d - y_pred_all)
    inside_tube = residuals <= eps_fixed + 1e-6
    n_sv  = (~inside_tube).sum()
    rmse  = np.sqrt(mean_squared_error(y_1d, y_pred_all))

    ax.scatter(X_1d_scaled[inside_tube],  y_1d[inside_tube],
               color='#bdc3c7', alpha=0.4, s=18)
    ax.scatter(X_1d_scaled[~inside_tube], y_1d[~inside_tube],
               color=col, s=50, edgecolors='black', linewidths=0.7,
               alpha=0.9, zorder=5, label=f'SVs ({n_sv})')

    ax.plot(X_grid, y_pred_grid, color=col, linewidth=2.5)
    ax.fill_between(X_grid.ravel(),
                    y_pred_grid - eps_fixed,
                    y_pred_grid + eps_fixed,
                    alpha=0.15, color=col, label=f'ε-tube (±{eps_fixed})')

    ax.set_title(f'C = {C}  |  SVs = {n_sv}  |  RMSE = {rmse:.3f}',
                 fontsize=12, fontweight='bold', color=col)
    ax.legend(fontsize=10)
    ax.set_xlabel('Word Count (scaled)', fontsize=10)

axes[0].set_ylabel('Response Time (hours)', fontsize=10)
axes[2].set_ylabel('Response Time (hours)', fontsize=10)

plt.suptitle(f'SVR (linear, ε={eps_fixed}) — Effect of C on Fit Tightness',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Summary:")
print(f"{'C':>8} | {'SVs':>5} | {'RMSE (train)':>14}")
print("-"*35)
for C in C_values:
    svr = SVR(kernel='linear', epsilon=eps_fixed, C=C)
    svr.fit(X_1d_scaled, y_1d)
    y_pred_all = svr.predict(X_1d_scaled)
    residuals  = np.abs(y_1d - y_pred_all)
    n_sv  = (residuals > eps_fixed + 1e-6).sum()
    rmse  = np.sqrt(mean_squared_error(y_1d, y_pred_all))
    print(f"{C:>8} | {n_sv:>5} | {rmse:>14.4f}")

In [ ]:
# =============================================================================
# SECTION 6: Linear vs RBF Kernel SVR
# =============================================================================
# The kernel trick lets SVR fit NONLINEAR relationships without changing the algorithm.
# Linear kernel: fits a straight line (hyperplane)
# RBF kernel:    fits a smooth nonlinear curve

# Create a slightly nonlinear 1D dataset for clearer visual contrast
np.random.seed(42)
X_nl = np.sort(np.random.uniform(-3, 3, 150))
y_nl = np.sin(X_nl) * 3 + 0.5 * X_nl + np.random.normal(0, 0.5, 150)

X_nl_2d  = X_nl.reshape(-1, 1)
X_nl_grid = np.linspace(-3, 3, 300).reshape(-1, 1)

svr_linear = SVR(kernel='linear', epsilon=0.3, C=1.0)
svr_rbf    = SVR(kernel='rbf',    epsilon=0.3, C=1.0, gamma='scale')
svr_poly   = SVR(kernel='poly',   epsilon=0.3, C=1.0, degree=3)

svr_linear.fit(X_nl_2d, y_nl)
svr_rbf.fit(X_nl_2d,    y_nl)
svr_poly.fit(X_nl_2d,   y_nl)

pred_linear = svr_linear.predict(X_nl_grid)
pred_rbf    = svr_rbf.predict(X_nl_grid)
pred_poly   = svr_poly.predict(X_nl_grid)

eps = 0.3
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

kernels = [
    ('Linear', pred_linear, svr_linear, '#e74c3c'),
    ('RBF',    pred_rbf,    svr_rbf,    '#2ecc71'),
    ('Poly(3)', pred_poly,  svr_poly,   '#3498db'),
]

for ax, (kname, pred, model, col) in zip(axes, kernels):
    y_tr     = model.predict(X_nl_2d)
    residuals = np.abs(y_nl - y_tr)
    inside   = residuals <= eps + 1e-6
    n_sv     = (~inside).sum()
    rmse     = np.sqrt(mean_squared_error(y_nl, y_tr))

    ax.scatter(X_nl[inside],  y_nl[inside],  color='#bdc3c7', s=20, alpha=0.5)
    ax.scatter(X_nl[~inside], y_nl[~inside], color=col, s=50,
               edgecolors='black', linewidths=0.7, alpha=0.9, zorder=5)
    ax.plot(X_nl_grid, pred, color=col, linewidth=2.5, label=f'{kname} SVR')
    ax.fill_between(X_nl_grid.ravel(), pred-eps, pred+eps,
                    alpha=0.15, color=col, label='ε-tube')

    # True curve
    X_true = np.linspace(-3, 3, 300)
    y_true = np.sin(X_true)*3 + 0.5*X_true
    ax.plot(X_true, y_true, 'k--', linewidth=1.5, alpha=0.5, label='True f(x)')

    ax.set_title(f'{kname} Kernel\nSVs={n_sv}  RMSE={rmse:.3f}',
                 fontsize=12, fontweight='bold', color=col)
    ax.legend(fontsize=9)
    ax.set_xlabel('Input x', fontsize=11)

axes[0].set_ylabel('Output y', fontsize=11)
plt.suptitle('SVR with Different Kernels — Fitting a Nonlinear Relationship',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key takeaway: RBF kernel can capture the sine-like nonlinearity that the linear kernel misses.")
print("The kernel trick applies the same SVR algorithm — only the notion of 'distance' changes.")

In [ ]:
# =============================================================================
# SECTION 7: SVR's Robustness to Outliers vs OLS
# =============================================================================
# This is one of SVR's most valuable properties in practice.
# OLS minimizes squared errors — an outlier 10x away contributes 100x the loss.
# SVR's ε-insensitive loss + linear penalty = outliers don't dominate the solution.

np.random.seed(42)

# Clean dataset: response time ~ 2 + 0.02*word_count
X_rob = np.sort(np.random.uniform(50, 500, 80)).reshape(-1, 1)
y_clean = 2 + 0.02 * X_rob.ravel() + np.random.normal(0, 0.8, 80)

# Add 5 extreme outliers (emails that took MUCH longer than expected)
X_outliers = np.array([[100], [200], [300], [400], [450]])
y_outliers  = np.array([18, 19, 17, 20, 18])   # extreme values

X_with_outliers = np.vstack([X_rob, X_outliers])
y_with_outliers = np.concatenate([y_clean, y_outliers])

# Scale
sc = StandardScaler()
X_clean_sc   = sc.fit_transform(X_rob)
X_outlier_sc = sc.transform(X_with_outliers)

X_grid_rob = np.linspace(X_outlier_sc.min(), X_outlier_sc.max(), 300).reshape(-1, 1)

# Models
ols_clean   = LinearRegression().fit(X_clean_sc,   y_clean)
ols_outlier = LinearRegression().fit(X_outlier_sc, y_with_outliers)
svr_clean   = SVR(kernel='linear', epsilon=0.5, C=1.0).fit(X_clean_sc,   y_clean)
svr_outlier = SVR(kernel='linear', epsilon=0.5, C=1.0).fit(X_outlier_sc, y_with_outliers)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, title in zip(axes, ['Without Outliers', 'With 5 Extreme Outliers']):
    if title == 'Without Outliers':
        X_sc, y_use = X_clean_sc, y_clean
    else:
        X_sc, y_use = X_outlier_sc, y_with_outliers

    ols = LinearRegression().fit(X_sc, y_use)
    svr = SVR(kernel='linear', epsilon=0.5, C=1.0).fit(X_sc, y_use)

    pred_ols = ols.predict(X_grid_rob)
    pred_svr = svr.predict(X_grid_rob)

    # Mark outliers
    if title == 'With 5 Extreme Outliers':
        n_clean = len(y_clean)
        ax.scatter(X_sc[:n_clean], y_use[:n_clean], color='#3498db', s=30,
                   alpha=0.6, label='Normal emails')
        ax.scatter(X_sc[n_clean:], y_use[n_clean:], color='#e74c3c', s=120,
                   marker='*', zorder=6, label='Outliers (extreme delays)', edgecolors='black')
    else:
        ax.scatter(X_sc, y_use, color='#3498db', s=30, alpha=0.6, label='Emails')

    ax.plot(X_grid_rob, pred_ols, color='#e74c3c', linewidth=2.5,
            linestyle='--', label='OLS regression')
    ax.plot(X_grid_rob, pred_svr, color='#2ecc71', linewidth=2.5,
            label='SVR (linear, ε=0.5)')
    ax.fill_between(X_grid_rob.ravel(), pred_svr - 0.5, pred_svr + 0.5,
                    alpha=0.15, color='#2ecc71')

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Word Count (scaled)', fontsize=11)
    ax.set_ylabel('Response Time (hours)', fontsize=11)
    ax.legend(fontsize=10)

plt.suptitle('SVR vs OLS: Robustness to Outliers', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Quantify the damage
sc2 = StandardScaler()
Xo_sc = sc2.fit_transform(X_outlier_sc)
ols_clean_pred   = LinearRegression().fit(sc.transform(X_rob),   y_clean).predict(X_grid_rob)
ols_outlier_pred = LinearRegression().fit(X_outlier_sc, y_with_outliers).predict(X_grid_rob)
svr_clean_pred   = SVR(kernel='linear', epsilon=0.5, C=1.0).fit(sc.transform(X_rob),   y_clean).predict(X_grid_rob)
svr_outlier_pred = SVR(kernel='linear', epsilon=0.5, C=1.0).fit(X_outlier_sc, y_with_outliers).predict(X_grid_rob)

ols_shift = np.abs(ols_outlier_pred - ols_clean_pred).mean()
svr_shift = np.abs(svr_outlier_pred - svr_clean_pred).mean()
print(f"Average prediction shift due to outliers:")
print(f"  OLS: {ols_shift:.3f} hours  (LARGE — pulled toward outliers)")
print(f"  SVR: {svr_shift:.3f} hours  (SMALL — robust to outliers)")

In [ ]:
# =============================================================================
# SECTION 8: ε Effect — Count Support Vectors and Scatter Predictions vs Actual
# =============================================================================

# Multivariate SVR on the full email dataset
X_full = df[['word_count', 'complexity_score', 'sender_importance']].values
y_full = df['response_time'].values

scaler_full = StandardScaler()
X_full_sc   = scaler_full.fit_transform(X_full)

X_tr, X_te, y_tr, y_te = train_test_split(X_full_sc, y_full, test_size=0.2, random_state=42)

epsilons_test = [0.05, 0.2, 0.5, 2.0]

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
sv_counts  = []
test_rmses = []

for col_idx, eps in enumerate(epsilons_test):
    svr_m = SVR(kernel='rbf', epsilon=eps, C=1.0, gamma='scale')
    svr_m.fit(X_tr, y_tr)

    y_pred_tr = svr_m.predict(X_tr)
    y_pred_te = svr_m.predict(X_te)

    n_sv       = svr_m.n_support_.sum()
    test_rmse  = np.sqrt(mean_squared_error(y_te, y_pred_te))
    sv_counts.append(n_sv)
    test_rmses.append(test_rmse)

    # Top row: scatter predicted vs actual
    ax_top = axes[0, col_idx]
    ax_top.scatter(y_te, y_pred_te, alpha=0.6, color='#3498db', s=30, edgecolors='none')
    lims = [min(y_te.min(), y_pred_te.min()) - 0.5,
            max(y_te.max(), y_pred_te.max()) + 0.5]
    ax_top.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
    ax_top.set_xlabel('Actual Response Time', fontsize=9)
    ax_top.set_ylabel('Predicted', fontsize=9)
    ax_top.set_title(f'ε={eps}\nRMSE={test_rmse:.3f}', fontsize=11, fontweight='bold')
    ax_top.legend(fontsize=8)

    # Bottom row: residuals
    ax_bot = axes[1, col_idx]
    resid = y_te - y_pred_te
    ax_bot.hist(resid, bins=25, color='#9b59b6', alpha=0.7, edgecolor='white')
    ax_bot.axvline(-eps, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'-ε={-eps}')
    ax_bot.axvline(+eps, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'+ε={eps}')
    ax_bot.axvspan(-eps, eps, alpha=0.1, color='#2ecc71')
    ax_bot.set_xlabel('Residual (actual - predicted)', fontsize=9)
    ax_bot.set_ylabel('Count', fontsize=9)
    ax_bot.set_title(f'SVs={n_sv}', fontsize=11, fontweight='bold')
    ax_bot.legend(fontsize=8)

plt.suptitle('SVR (RBF, C=1): Effect of ε on Support Vectors and Prediction Quality',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"{'ε':>6} | {'Support Vectors':>17} | {'Test RMSE':>10}")
print("-"*40)
for eps, nsv, rmse in zip(epsilons_test, sv_counts, test_rmses):
    print(f"{eps:>6} | {nsv:>17} | {rmse:>10.4f}")

In [ ]:
# =============================================================================
# SECTION 9: Support Vectors Highlighted in Regression Context
# =============================================================================
# Back to 1D to clearly show which points are SVs (tube violators)

np.random.seed(42)

eps_show = 0.8
svr_show = SVR(kernel='linear', epsilon=eps_show, C=2.0)
svr_show.fit(X_1d_scaled, y_1d)

y_pred_all = svr_show.predict(X_1d_scaled)
residuals  = np.abs(y_1d - y_pred_all)
is_sv      = residuals > eps_show + 1e-6  # outside tube = support vector

y_pred_grid_show = svr_show.predict(X_grid)

fig, ax = plt.subplots(figsize=(13, 7))

# Inside tube — gray, small
ax.scatter(X_1d_scaled[~is_sv].ravel(), y_1d[~is_sv],
           color='#bdc3c7', s=25, alpha=0.5, label=f'Inside tube ({(~is_sv).sum()} points, α=0)',
           zorder=2)

# Support vectors — colored, ringed
ax.scatter(X_1d_scaled[is_sv].ravel(), y_1d[is_sv],
           color='#e74c3c', s=80, edgecolors='#c0392b', linewidths=1.5,
           label=f'Support Vectors ({is_sv.sum()} points, α≠0)',
           zorder=5)

# Error lines from support vectors to tube boundary
for xi, yi, ypi in zip(X_1d_scaled[is_sv].ravel(), y_1d[is_sv], y_pred_all[is_sv]):
    if yi > ypi:  # above tube
        ax.plot([xi, xi], [ypi + eps_show, yi], color='#e74c3c', linewidth=0.8, alpha=0.5, zorder=3)
    else:          # below tube
        ax.plot([xi, xi], [ypi - eps_show, yi], color='#e74c3c', linewidth=0.8, alpha=0.5, zorder=3)

# SVR line and tube
ax.plot(X_grid, y_pred_grid_show, color='#2c3e50', linewidth=2.5, label='SVR regression line', zorder=4)
ax.fill_between(X_grid.ravel(),
                y_pred_grid_show - eps_show,
                y_pred_grid_show + eps_show,
                alpha=0.2, color='#3498db',
                label=f'ε-tube (±{eps_show} hours)')
ax.plot(X_grid, y_pred_grid_show - eps_show, 'b--', linewidth=1.2, alpha=0.6)
ax.plot(X_grid, y_pred_grid_show + eps_show, 'b--', linewidth=1.2, alpha=0.6)

ax.set_xlabel('Word Count (standardized)', fontsize=12)
ax.set_ylabel('Email Response Time (hours)', fontsize=12)
ax.set_title(
    f'SVR Support Vectors in Regression\n'
    f'Only tube-violating points (red) have α≠0 and shape the regression line',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=11, loc='upper left')

# Annotations
ax.text(0.02, 0.92, f'Total points: {len(y_1d)}', transform=ax.transAxes, fontsize=11)
ax.text(0.02, 0.87, f'Inside tube (α=0): {(~is_sv).sum()}', transform=ax.transAxes,
        fontsize=11, color='gray')
ax.text(0.02, 0.82, f'Support vectors (α≠0): {is_sv.sum()}', transform=ax.transAxes,
        fontsize=11, color='#e74c3c')
ax.text(0.02, 0.77,
        f'→ Model determined by only {is_sv.sum()}/{len(y_1d)} = {100*is_sv.mean():.1f}% of data!',
        transform=ax.transAxes, fontsize=11, color='#2c3e50', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SECTION 10: Full Pipeline — Multivariate SVR with Cross-Validation
# =============================================================================
from sklearn.model_selection import cross_val_score, GridSearchCV

print("Multivariate SVR on Email Response Time Dataset")
print("="*55)
print(f"Features: word_count, complexity_score, sender_importance")
print(f"Target:   response_time (hours)")
print(f"Samples:  {len(df)}")
print()

# Compare models
models = {
    'OLS Linear':          LinearRegression(),
    'SVR (linear, ε=0.5)': SVR(kernel='linear', epsilon=0.5,  C=1.0),
    'SVR (rbf, ε=0.5)':    SVR(kernel='rbf',    epsilon=0.5,  C=1.0, gamma='scale'),
    'SVR (rbf, ε=0.2)':    SVR(kernel='rbf',    epsilon=0.2,  C=5.0, gamma='scale'),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_full_sc, y_full,
                              cv=5, scoring='neg_root_mean_squared_error')
    results[name] = {'mean': -scores.mean(), 'std': scores.std()}
    print(f"{name:30s}: CV RMSE = {-scores.mean():.4f} ± {scores.std():.4f}")

print()

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
names  = list(results.keys())
means  = [results[n]['mean'] for n in names]
stds   = [results[n]['std']  for n in names]
colors_bar = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

bars = ax.barh(names, means, xerr=stds, color=colors_bar, alpha=0.8,
               edgecolor='black', linewidth=0.8, capsize=5)

for bar, mean in zip(bars, means):
    ax.text(mean + 0.01, bar.get_y() + bar.get_height()/2,
            f'{mean:.4f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('CV RMSE (lower is better)', fontsize=12)
ax.set_title('5-Fold Cross-Validation RMSE: SVR vs OLS on Email Response Time',
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## SVR Dual Formulation and the Kernel Trick

Just like classification SVMs, SVR has a **dual problem** that enables the kernel trick.

**SVR Dual (maximise):**
$$W(\alpha, \alpha^*) = -\frac{1}{2}\sum_{i,j}(\alpha_i - \alpha_i^*)(\alpha_j - \alpha_j^*)K(x_i,x_j) - \varepsilon\sum_i(\alpha_i + \alpha_i^*) + \sum_i y_i(\alpha_i - \alpha_i^*)$$

**Subject to:**
$$\sum_i (\alpha_i - \alpha_i^*) = 0, \quad 0 \leq \alpha_i, \alpha_i^* \leq C$$

**The regression function in dual form:**
$$f(x) = \sum_{i \in SV} (\alpha_i - \alpha_i^*) K(x_i, x) + b$$

**Key observations:**
- We only need kernel evaluations $K(x_i, x_j)$ — never the explicit feature vectors in high-dimensional space
- For a point **inside** the tube: $\alpha_i = \alpha_i^* = 0$ (not a support vector)
- For a point **above** the tube: $\alpha_i > 0$, $\alpha_i^* = 0$
- For a point **below** the tube: $\alpha_i = 0$, $\alpha_i^* > 0$
- No point can have both $\alpha_i > 0$ and $\alpha_i^* > 0$ simultaneously (complementary slackness)

**This is why SVR is sparse:** Only the $|SV|$ support vectors (tube violators) appear in the prediction sum. All other training points contribute nothing.

In [ ]:
# =============================================================================
# SECTION 11: SVR Dual Coefficients — Inspecting Support Vector Weights
# =============================================================================
# sklearn SVR exposes the dual coefficients: (alpha_i - alpha_i*) for each SV.
# Positive coef → SV is ABOVE the tube (actual > predicted + ε)
# Negative coef → SV is BELOW the tube (actual < predicted - ε)

svr_inspect = SVR(kernel='rbf', epsilon=0.5, C=1.0, gamma='scale')
svr_inspect.fit(X_tr, y_tr)

dual_coefs = svr_inspect.dual_coef_[0]   # shape: (n_svs,)

print("SVR (RBF, epsilon=0.5, C=1.0) — Dual Coefficient Analysis")
print("="*55)
print(f"Training samples:    {len(X_tr)}")
print(f"Support vectors:     {svr_inspect.n_support_[0]}  "
      f"({100*svr_inspect.n_support_[0]/len(X_tr):.1f}% of training data)")
print(f"Dual coef range:     [{dual_coefs.min():.4f}, {dual_coefs.max():.4f}]")
print(f"  Positive (above tube): {(dual_coefs > 0).sum()} SVs")
print(f"  Negative (below tube): {(dual_coefs < 0).sum()} SVs")
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of dual coefficients
ax = axes[0]
ax.hist(dual_coefs[dual_coefs > 0], bins=20, color='#e74c3c', alpha=0.75,
        edgecolor='white', label='Positive (alpha_i > 0): above tube')
ax.hist(dual_coefs[dual_coefs < 0], bins=20, color='#3498db', alpha=0.75,
        edgecolor='white', label='Negative (alpha*_i > 0): below tube')
ax.axvline(0, color='black', linewidth=2)
ax.set_xlabel('Dual Coefficient (alpha_i minus alpha*_i)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of SVR Dual Coefficients\n(non-SVs have coefficient = 0, not shown)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Right: actual vs predicted for test set, color-coded by tube status
ax2 = axes[1]
y_pred_te2 = svr_inspect.predict(X_te)
residuals_te = y_te - y_pred_te2
inside_test  = np.abs(residuals_te) <= 0.5 + 1e-6

ax2.scatter(y_te[inside_test],  y_pred_te2[inside_test],
            color='#2ecc71', s=40, alpha=0.7, label=f'Within epsilon-tube ({inside_test.sum()})')
ax2.scatter(y_te[~inside_test], y_pred_te2[~inside_test],
            color='#e74c3c', s=60, alpha=0.8, edgecolors='black', linewidths=0.5,
            label=f'Outside epsilon-tube ({(~inside_test).sum()})')
lim = [y_te.min()-0.5, y_te.max()+0.5]
ax2.plot(lim, lim, 'k--', linewidth=1.5, label='Perfect prediction')
ax2.plot(lim, [l+0.5 for l in lim], 'b:', linewidth=1.2, alpha=0.6, label='+epsilon boundary')
ax2.plot(lim, [l-0.5 for l in lim], 'b:', linewidth=1.2, alpha=0.6, label='-epsilon boundary')
ax2.set_xlabel('Actual Response Time (hours)', fontsize=11)
ax2.set_ylabel('Predicted Response Time (hours)', fontsize=11)
ax2.set_title('Test Set Predictions\n(green = inside tube, red = outside)',
              fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('SVR Dual Coefficient Inspection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Reminder: The SVR prediction formula is:")
print("  f(x) = sum_{i in SV} (alpha_i - alpha*_i) * K(x_i, x) + b")
print("  Only support vectors (outside the tube) have nonzero coefficients.")

In [ ]:
# =============================================================================
# SECTION 12: Feature Importance in SVR (Linear Weights + Permutation)
# =============================================================================
# Linear SVR: weight vector w gives direct feature importance.
# Nonlinear SVR: use permutation importance (model-agnostic).

from sklearn.inspection import permutation_importance

feature_names = ['word_count', 'complexity_score', 'sender_importance']

# Linear SVR — inspect w directly (coef_ attribute)
svr_lin_full = SVR(kernel='linear', epsilon=0.5, C=1.0)
svr_lin_full.fit(X_tr, y_tr)
w_linear = svr_lin_full.coef_[0]

# Permutation importance for RBF SVR
perm_imp = permutation_importance(
    svr_inspect, X_te, y_te,
    n_repeats=20, random_state=42,
    scoring='neg_root_mean_squared_error'
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Linear SVR weights
ax = axes[0]
colors_feat = ['#e74c3c' if w > 0 else '#3498db' for w in w_linear]
bars = ax.barh(feature_names, np.abs(w_linear), color=colors_feat, alpha=0.85,
               edgecolor='black', linewidth=0.7)
for bar, w in zip(bars, w_linear):
    sign = '+' if w > 0 else '-'
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{sign}{abs(w):.4f}', va='center', fontsize=11)
ax.set_xlabel('|Weight| (standardized features)', fontsize=11)
ax.set_title('Linear SVR — Feature Weights\n(red=positive effect, blue=negative effect on response time)',
             fontsize=11, fontweight='bold')
ax.set_xlim(0, max(np.abs(w_linear)) * 1.5)

# Right: Permutation importance for RBF SVR
ax2 = axes[1]
sorted_idx = perm_imp.importances_mean.argsort()[::-1]
ax2.barh([feature_names[i] for i in sorted_idx],
         perm_imp.importances_mean[sorted_idx],
         xerr=perm_imp.importances_std[sorted_idx],
         color='#9b59b6', alpha=0.8, edgecolor='black', linewidth=0.7, capsize=5)
ax2.set_xlabel('Mean RMSE increase when feature permuted\n(higher = more important)', fontsize=10)
ax2.set_title('RBF SVR — Permutation Importance\n(20 repeats, test set)',
              fontsize=11, fontweight='bold')

plt.suptitle('SVR Feature Importance Analysis\n(Email Response Time Prediction)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Linear SVR feature weights (standardized features):")
for name, w in zip(feature_names, w_linear):
    direction = 'longer response' if w > 0 else 'shorter response'
    print(f"  {name:22s}: w={w:+.4f}  => {direction}")
print()
print("Permutation importance (RBF SVR):")
for i in sorted_idx:
    print(f"  {feature_names[i]:22s}: {perm_imp.importances_mean[i]:.4f} +/- {perm_imp.importances_std[i]:.4f}")

In [ ]:
# =============================================================================
# SECTION 13: Learning Curves — SVR vs OLS as Training Size Grows
# =============================================================================
from sklearn.model_selection import learning_curve

train_sizes_pct = np.linspace(0.1, 1.0, 10)

models_lc = {
    'SVR (RBF, epsilon=0.5)': SVR(kernel='rbf', epsilon=0.5, C=1.0, gamma='scale'),
    'OLS Linear':              LinearRegression(),
}
colors_lc = ['#2ecc71', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model), col in zip(axes, models_lc.items(), colors_lc):
    train_sz, train_sc, val_sc = learning_curve(
        model, X_full_sc, y_full,
        train_sizes=train_sizes_pct,
        cv=5,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    train_mean = -train_sc.mean(axis=1)
    train_std  =  train_sc.std(axis=1)
    val_mean   = -val_sc.mean(axis=1)
    val_std    =  val_sc.std(axis=1)

    ax.plot(train_sz, train_mean, 'o-', color=col,       linewidth=2.5, markersize=7,
            label='Training RMSE')
    ax.plot(train_sz, val_mean,   's-', color='#2c3e50', linewidth=2.5, markersize=7,
            label='Validation RMSE (CV)')
    ax.fill_between(train_sz, train_mean-train_std, train_mean+train_std, alpha=0.12, color=col)
    ax.fill_between(train_sz, val_mean-val_std,     val_mean+val_std,     alpha=0.12, color='#2c3e50')

    gap_final = val_mean[-1] - train_mean[-1]
    ax.set_title(f'{name}\nFinal gap (val - train): {gap_final:.4f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Training Set Size', fontsize=11)
    ax.set_ylabel('RMSE (hours)', fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim(0)
    ax.axhline(val_mean[-1], color=col, linestyle=':', linewidth=1.2, alpha=0.5)

plt.suptitle('Learning Curves: SVR vs OLS on Email Response Time\n'
             'Gap between curves = variance; height of both = bias',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Learning curve interpretation:")
print("  Large gap between train and val curves → HIGH VARIANCE (overfitting)")
print("  Both curves high → HIGH BIAS (underfitting)")
print("  Both curves converging low → GOOD FIT")
print()
print("  SVR note: the training RMSE may be higher than expected because")
print("  SVR deliberately ignores small errors (inside the epsilon-tube).")

## When to Use SVR vs OLS vs Other Regressors

| Situation | Recommended Model | Reason |
|-----------|------------------|--------|
| Small noise, linear relationship | OLS | Fast, interpretable, closed-form |
| Noisy target, tolerance for small errors | SVR (linear) | epsilon-tube absorbs noise; sparse |
| Outliers in target variable | SVR | Linear penalty bounds outlier influence |
| Nonlinear relationship, n < 10K | SVR (RBF) | Kernel trick, no feature engineering |
| n > 50K | Linear SVR or Ridge | SVR training is O(n²–n³) |
| Very large n (millions) | LinearSVC / SGD | LIBLINEAR, coordinate descent |
| Interpretability required | OLS or Linear SVR | Coefficients directly readable |
| Uncertainty quantification | Gaussian Process Regression | Gives confidence intervals |

**SVR Practical Checklist:**

1. Scale your features first (`StandardScaler`) — SVR is sensitive to feature scales
2. Set epsilon based on your domain tolerance (e.g., ε=1 hour if ±1 hour response time is acceptable)
3. Use RBF kernel as default; try linear if n_features >> n_samples
4. Grid search C and epsilon jointly (they interact: large ε needs larger C to fit well)
5. Monitor the number of support vectors — if > 70% of training data, ε is too small
6. Use cross-validation for all hyperparameters; never tune on the test set

**Common mistakes:**
- Not scaling features (leads to poor results and slow convergence)
- Setting epsilon = 0 (becomes equivalent to hard-margin — very sensitive)
- Using kernel='rbf' with unscaled features (gamma='scale' helps but scaling is still essential)
- Comparing train RMSE to OLS train RMSE — SVR's training RMSE is higher by design (epsilon-insensitivity)

## Summary: SVR Key Concepts

| Concept | OLS Regression | SVR |
|---------|---------------|-----|
| Loss function | MSE — all errors penalized | ε-insensitive — small errors free |
| Sensitivity to noise | High (even tiny noise moves the line) | Low (absorbed by ε-tube) |
| Sensitivity to outliers | High (squared penalty grows fast) | Low (linear penalty, C controls it) |
| Solution sparsity | Dense (all points matter) | Sparse (only SVs outside tube matter) |
| Nonlinear extension | Requires feature engineering | Kernel trick (no feature engineering) |
| Key hyperparameters | None (closed-form) | C, ε, kernel, kernel params |

**Hyperparameter rules of thumb:**
- **ε**: Start with ~10-20% of target std. Tune based on how much noise you can tolerate.
- **C**: Use cross-validation. Try logarithmic grid: 0.01, 0.1, 1, 10, 100.
- **Kernel**: Start with RBF (works for most problems). Use linear if n_features >> n_samples.

**When to use SVR:**
- Target variable has noise you want to ignore (prediction within ε is "good enough")
- Dataset has outliers in the target variable
- You need a nonlinear regression without explicit feature engineering
- n < 10,000 (SVR scales as O(n²) to O(n³) for training)

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** SVR with ε=5 predicts response time perfectly within ±5 hours for 80% of emails. Those 80% contribute zero to the training loss. Does this mean the model doesn't **learn** from them?

<details>
<summary>Click to reveal answer</summary>

**No — the model does learn from inside-tube points, just not through the loss gradient.**

The 80% of inside-tube points have $\alpha_i = 0$ in the dual formulation, meaning they contribute zero to the weight vector **w** and the regression function f(x). However, they still influence the model **implicitly**: the model must find a function that places these 80% of points inside the tube. The tube position is determined by the 20% of support vectors (outside-tube points), but the tube must be wide enough (ε=5) to contain those 80%.

More precisely: during training optimization, the inside-tube points define the **feasibility constraints** (they must satisfy yᵢ - f(xᵢ) ≤ ε with slack = 0). The support vectors (outside the tube) determine **where** the regression line sits.

**Analogy:** In the plumber example, the 80% of on-budget projects don't change how the plumber quotes jobs (they don't affect the model). But the 20% of overbudget projects — the support vectors — are exactly what shapes how the plumber estimates future costs.

</details>

---

**Question 2:** You increase ε from 0.1 to 5.0. The number of support vectors drops from 280 to 15. Is the model with ε=5.0 better or worse? What must you check?

<details>
<summary>Click to reveal answer</summary>

**You cannot tell from the number of SVs alone — you must check generalization (test set performance).**

- With ε=0.1: 280 SVs means almost every point is outside the narrow tube. The model is complex, shaped by nearly all training points. It may **overfit** to noise.
- With ε=5.0: Only 15 SVs. The model is very simple — only the 15 worst mispredictions shape it. Predictions within 5 hours of truth are considered "correct." For some problems this is too coarse (**underfitting**).

**What to check:**
1. **Test set RMSE**: Does the simpler model (ε=5.0) generalize better?
2. **Business requirement**: Is ±5 hours a tolerable prediction error for your application? If you need ±0.5 hours accuracy, ε=5.0 is useless regardless of RMSE.
3. **Bias-variance tradeoff**: Large ε → high bias, low variance. Small ε → low bias, high variance. Cross-validate both.

**Rule of thumb:** Set ε based on what constitutes a "good enough" prediction for your use case, then tune C for the best generalization within that ε.

</details>

---

**Question 3:** SVR with large C fits training data very closely. OLS also does this (especially with many features). What makes SVR with large C **different** from OLS?

<details>
<summary>Click to reveal answer</summary>

**Three key differences:**

1. **Loss function**: Even with large C, SVR still uses ε-insensitive loss — errors within ε cost nothing. OLS uses MSE — every error (even 0.0001) is penalized. With large C, SVR tries very hard to push all points inside the tube, but once inside, it ignores them. OLS never ignores any error.

2. **Outlier handling**: SVR with large C will try hard to accommodate outliers (put them inside the tube), but the penalty is **linear** in the violation amount. OLS gives outliers **quadratic** penalty — a point 3x farther from the line has 9x the influence. SVR's linear penalty means outliers have bounded influence even at large C.

3. **Kernel and implicit feature space**: SVR can use kernels (RBF, polynomial) with large C, fitting complex nonlinear relationships that OLS cannot capture without explicit feature engineering. OLS is always linear in the original feature space.

**In summary:** SVR with large C approximates a minimum-||w|| function that passes through (or very near) training points — analogous to OLS with regularization. But the ε-insensitive loss and kernel flexibility make it fundamentally different from OLS in how it treats errors and what function classes it can represent.

</details>

In [ ]:
# =============================================================================
# BONUS: SVR Hyperparameter Grid Search on Email Dataset
# =============================================================================
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C':       [0.1, 1.0, 10.0],
    'epsilon': [0.1, 0.5, 1.0],
    'gamma':   ['scale', 'auto']
}

svr_gs = GridSearchCV(
    SVR(kernel='rbf'),
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=0
)
svr_gs.fit(X_full_sc, y_full)

print("Grid Search Results (RBF SVR on Email Response Time)")
print("="*52)
print(f"Best parameters:  {svr_gs.best_params_}")
print(f"Best CV RMSE:     {-svr_gs.best_score_:.4f} hours")
print()

# Heatmap of C vs epsilon (averaged over gamma)
results_df = pd.DataFrame(svr_gs.cv_results_)
pivot = results_df.groupby(['param_C', 'param_epsilon'])['mean_test_score'].mean().unstack()
pivot = -pivot  # convert from negative RMSE

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd_r',
            xticklabels=pivot.columns, yticklabels=pivot.index, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_xlabel('ε (epsilon)', fontsize=12)
ax.set_ylabel('C', fontsize=12)
ax.set_title('Grid Search: CV RMSE for C × ε (RBF SVR)\nLower = better', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  Rows = C (penalty for tube violations)")
print("  Cols = ε (tube half-width)")
print("  Cell = mean CV RMSE (lower is better)")
print(f"  Best: C={svr_gs.best_params_['C']}, ε={svr_gs.best_params_['epsilon']}")

## Key Takeaways

1. **SVR defines an ε-tube**: Predictions within ε of the truth have zero loss. This makes the model robust to noise and produces a sparse solution (only tube-violating points matter).

2. **Support vectors in regression are the tube violators**: Points inside the tube have $\alpha_i = 0$ and don't contribute to the model. Only outside-tube points shape the regression function.

3. **ε controls complexity**: Wide ε → fewer SVs → simpler model. Narrow ε → more SVs → complex model. Set ε based on your business tolerance for error.

4. **C controls fit tightness**: Large C → penalize violations → tight fit. Small C → allow violations → smooth function.

5. **SVR is robust to outliers**: Linear penalty (not quadratic like OLS) means outliers have bounded influence.

6. **Kernel trick extends SVR to nonlinear regression**: Same algorithm, different notion of similarity. RBF kernel handles most nonlinear problems.

---

**Next:** Notebook 10 — The SMO Algorithm: How is SVR/SVM actually optimized at scale?